# Notebook 04 — V4: La trampa de la igualdad formal (scatter)

**Visualización:** Scatter plot en Flourish  
**Pregunta:** ¿Los países con mayor igualdad formal (EIGE) tienen menor brecha de bienestar? ¿O la igualdad formal no garantiza igualdad de bienestar?  
**Output:** `data/processed/v4_scatter_nordica.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../')
OUT  = ROOT / 'data' / 'processed'
EIGE_PATH = ROOT / 'EIGE-scores' / 'index_scores_Data.csv'

# Tabla de correspondencia ESS (ISO 2) ↔ EIGE (nombre completo)
PAISES_ESS_EIGE = {
    'AT': 'Austria',      'BE': 'Belgium',     'BG': 'Bulgaria',
    'HR': 'Croatia',      'CY': 'Cyprus',      'CZ': 'Czechia',
    'DK': 'Denmark',      'EE': 'Estonia',     'FI': 'Finland',
    'FR': 'France',       'DE': 'Germany',     'GR': 'Greece',
    'HU': 'Hungary',      'IE': 'Ireland',     'IT': 'Italy',
    'LV': 'Latvia',       'LT': 'Lithuania',   'LU': 'Luxembourg',
    'MT': 'Malta',        'NL': 'Netherlands', 'PL': 'Poland',
    'PT': 'Portugal',     'RO': 'Romania',     'SK': 'Slovakia',
    'SI': 'Slovenia',     'ES': 'Spain',       'SE': 'Sweden'
}

print('Setup OK')

Setup OK


## 1. Cargar y preparar BRECHA_GNDR desde ESS R11

In [2]:
ess11 = pd.read_csv(OUT / 'ESS11_clean.csv', low_memory=False)

# Calcular bienestar medio por país y género
wb = (
    ess11.groupby(['cntry', 'gndr'])['IDX_WELLBEING']
    .agg(mean='mean', n='count')
    .reset_index()
)
wb_wide = wb.pivot(index='cntry', columns='gndr', values=['mean', 'n'])
wb_wide.columns = ['wb_hombres', 'wb_mujeres', 'n_hombres', 'n_mujeres']
wb_wide = wb_wide.reset_index()
wb_wide['BRECHA_GNDR'] = (wb_wide['wb_hombres'] - wb_wide['wb_mujeres']).round(4)
wb_wide['n_total'] = wb_wide['n_hombres'] + wb_wide['n_mujeres']

# Solo países del ESS que tienen equivalente en EIGE (EU27)
wb_eu = wb_wide[wb_wide['cntry'].isin(PAISES_ESS_EIGE.keys())].copy()
wb_eu['country_name'] = wb_eu['cntry'].map(PAISES_ESS_EIGE)

print(f'Países ESS con BRECHA_GNDR: {len(wb_wide)}')
print(f'Países ESS en EU27 (para merge con EIGE): {len(wb_eu)}')
print(f'Países ESS no en EU27 (excluidos): {sorted(set(wb_wide["cntry"]) - set(PAISES_ESS_EIGE.keys()))}')

Países ESS con BRECHA_GNDR: 30
Países ESS en EU27 (para merge con EIGE): 22
Países ESS no en EU27 (excluidos): ['CH', 'GB', 'IL', 'IS', 'ME', 'NO', 'RS', 'UA']


## 2. Cargar y preparar índice EIGE

In [3]:
eige_raw = pd.read_csv(EIGE_PATH)
print(f'EIGE cargado: {eige_raw.shape}')
print(f'Años: {eige_raw["Time"].unique()}')
print(f'Países: {sorted(eige_raw["Geographic region"].unique())}')

EIGE cargado: (560, 4)
Años: [2025]
Países: ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'European Union - 27 countries (from 2020)', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden']


In [4]:
# Filtrar solo el índice global
eige_overall = eige_raw[
    eige_raw['(Sub-) Domain Scores'] == 'Overall Gender Equality Index'
][['Geographic region', 'Value']].copy()
eige_overall.columns = ['country_name', 'eige_index']

# Excluir la fila de la UE agregada
eige_overall = eige_overall[
    eige_overall['country_name'] != 'European Union - 27 countries (from 2020)'
].copy()

print(f'Países en EIGE Overall: {len(eige_overall)}')
print(eige_overall.sort_values('eige_index', ascending=False).to_string(index=False))

Países en EIGE Overall: 27
country_name  eige_index
      Sweden        73.7
      France        73.4
     Denmark        71.8
       Spain        70.9
 Netherlands        69.5
     Ireland        69.0
     Belgium        68.5
     Finland        68.3
  Luxembourg        63.9
    Portugal        63.4
     Germany        63.2
       Italy        61.9
     Austria        61.2
   Lithuania        60.9
     Estonia        59.4
       Malta        58.9
    Bulgaria        58.1
    Slovenia        58.0
      Poland        57.8
    Slovakia        57.2
     Croatia        57.1
      Greece        57.0
     Romania        57.0
      Latvia        56.7
     Czechia        53.2
     Hungary        51.6
      Cyprus        47.6


## 3. Merge ESS ↔ EIGE

In [5]:
v4 = wb_eu.merge(eige_overall, on='country_name', how='inner')

print(f'Países tras merge: {len(v4)}')
paises_solo_ess  = set(wb_eu['country_name']) - set(eige_overall['country_name'])
paises_solo_eige = set(eige_overall['country_name']) - set(wb_eu['country_name'])
if paises_solo_ess:
    print(f'En ESS pero no en EIGE: {paises_solo_ess}')
if paises_solo_eige:
    print(f'En EIGE pero no en ESS R11: {paises_solo_eige}')

Países tras merge: 22
En EIGE pero no en ESS R11: {'Luxembourg', 'Denmark', 'Romania', 'Malta', 'Czechia'}


## 4. Calcular cuadrantes

In [6]:
mediana_eige   = v4['eige_index'].median()
mediana_brecha = v4['BRECHA_GNDR'].median()

print(f'Mediana EIGE index:   {mediana_eige:.1f}')
print(f'Mediana BRECHA_GNDR:  {mediana_brecha:.3f}')

def cuadrante(row):
    alta_igualdad = row['eige_index']   >= mediana_eige
    alta_brecha   = row['BRECHA_GNDR']  >= mediana_brecha
    if alta_igualdad and alta_brecha:
        return 'alta_igualdad_alta_brecha'     # la trampa
    elif alta_igualdad and not alta_brecha:
        return 'alta_igualdad_baja_brecha'     # lo esperado
    elif not alta_igualdad and alta_brecha:
        return 'baja_igualdad_alta_brecha'     # doble problema
    else:
        return 'baja_igualdad_baja_brecha'     # baja igualdad, brecha contenida

v4['cuadrante'] = v4.apply(cuadrante, axis=1)
v4['es_espana'] = v4['cntry'] == 'ES'

print()
print('Países por cuadrante:')
for q, grp in v4.groupby('cuadrante'):
    print(f'  {q}: {sorted(grp["cntry"].tolist())}')

Mediana EIGE index:   61.0
Mediana BRECHA_GNDR:  0.105

Países por cuadrante:
  alta_igualdad_alta_brecha: ['BE', 'ES', 'FR', 'IT', 'NL', 'PT']
  alta_igualdad_baja_brecha: ['AT', 'DE', 'FI', 'IE', 'SE']
  baja_igualdad_alta_brecha: ['CY', 'GR', 'HR', 'SI', 'SK']
  baja_igualdad_baja_brecha: ['BG', 'EE', 'HU', 'LT', 'LV', 'PL']


## 5. Exploración: correlación y patrones

In [7]:
corr = v4['eige_index'].corr(v4['BRECHA_GNDR'])
print(f'Correlación Pearson EIGE ↔ BRECHA_GNDR: {corr:.3f}')
print()
print('Tabla completa (ordenada por EIGE desc):')
cols_show = ['cntry', 'country_name', 'BRECHA_GNDR', 'eige_index', 'cuadrante', 'es_espana']
print(v4[cols_show].sort_values('eige_index', ascending=False).to_string(index=False))

Correlación Pearson EIGE ↔ BRECHA_GNDR: -0.134

Tabla completa (ordenada por EIGE desc):
cntry country_name  BRECHA_GNDR  eige_index                 cuadrante  es_espana
   SE       Sweden       0.0629        73.7 alta_igualdad_baja_brecha      False
   FR       France       0.1673        73.4 alta_igualdad_alta_brecha      False
   ES        Spain       0.2050        70.9 alta_igualdad_alta_brecha       True
   NL  Netherlands       0.1249        69.5 alta_igualdad_alta_brecha      False
   IE      Ireland      -0.0472        69.0 alta_igualdad_baja_brecha      False
   BE      Belgium       0.1701        68.5 alta_igualdad_alta_brecha      False
   FI      Finland       0.0023        68.3 alta_igualdad_baja_brecha      False
   PT     Portugal       0.2812        63.4 alta_igualdad_alta_brecha      False
   DE      Germany       0.0200        63.2 alta_igualdad_baja_brecha      False
   IT        Italy       0.1739        61.9 alta_igualdad_alta_brecha      False
   AT      Austria  

In [8]:
print('Cuadrante "la trampa" (alta igualdad + alta brecha):')
trampa = v4[v4['cuadrante'] == 'alta_igualdad_alta_brecha']
print(trampa[['cntry', 'country_name', 'eige_index', 'BRECHA_GNDR']]
      .sort_values('eige_index', ascending=False).to_string(index=False))
print()
print('Cuadrante "lo esperado" (alta igualdad + baja brecha):')
esperado = v4[v4['cuadrante'] == 'alta_igualdad_baja_brecha']
print(esperado[['cntry', 'country_name', 'eige_index', 'BRECHA_GNDR']]
      .sort_values('eige_index', ascending=False).to_string(index=False))
print()
print('España:')
print(v4[v4['cntry']=='ES'][cols_show].to_string(index=False))

Cuadrante "la trampa" (alta igualdad + alta brecha):
cntry country_name  eige_index  BRECHA_GNDR
   FR       France        73.4       0.1673
   ES        Spain        70.9       0.2050
   NL  Netherlands        69.5       0.1249
   BE      Belgium        68.5       0.1701
   PT     Portugal        63.4       0.2812
   IT        Italy        61.9       0.1739

Cuadrante "lo esperado" (alta igualdad + baja brecha):
cntry country_name  eige_index  BRECHA_GNDR
   SE       Sweden        73.7       0.0629
   IE      Ireland        69.0      -0.0472
   FI      Finland        68.3       0.0023
   DE      Germany        63.2       0.0200
   AT      Austria        61.2       0.0655

España:
cntry country_name  BRECHA_GNDR  eige_index                 cuadrante  es_espana
   ES        Spain        0.205        70.9 alta_igualdad_alta_brecha       True


## 6. Formato final y guardar

In [9]:
v4_out = v4[['cntry', 'country_name', 'BRECHA_GNDR', 'eige_index',
             'wb_hombres', 'wb_mujeres', 'n_total', 'cuadrante', 'es_espana']].copy()

for col in ['BRECHA_GNDR', 'wb_hombres', 'wb_mujeres', 'eige_index']:
    v4_out[col] = v4_out[col].round(3)

v4_out = v4_out.sort_values('eige_index', ascending=False).reset_index(drop=True)

def verificar_csv_flourish(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Filas: {len(df)} | Columnas: {list(df.columns)}')
    nulos = df.isnull().sum()
    nulos_con = nulos[nulos > 0]
    if len(nulos_con):
        print(f'Valores nulos:\n{nulos_con}')
    else:
        print('Sin valores nulos')
    assert not df.isin([float('inf'), float('-inf')]).any().any(), 'Hay infinitos'
    assert not df.isnull().all(axis=1).any(), 'Hay filas completamente nulas'
    print(f'Muestra:\n{df.head(3).to_string()}')
    print('✓ OK')

verificar_csv_flourish(v4_out, 'v4_scatter_nordica')

out_path = OUT / 'v4_scatter_nordica.csv'
v4_out.to_csv(out_path, index=False)
print(f'\nGuardado: {out_path}')


=== v4_scatter_nordica ===
Filas: 22 | Columnas: ['cntry', 'country_name', 'BRECHA_GNDR', 'eige_index', 'wb_hombres', 'wb_mujeres', 'n_total', 'cuadrante', 'es_espana']
Sin valores nulos
Muestra:
  cntry country_name  BRECHA_GNDR  eige_index  wb_hombres  wb_mujeres  n_total                  cuadrante  es_espana
0    SE       Sweden        0.063        73.7       6.605       6.542   1230.0  alta_igualdad_baja_brecha      False
1    FR       France        0.167        73.4       6.522       6.355   1769.0  alta_igualdad_alta_brecha      False
2    ES        Spain        0.205        70.9       6.562       6.357   1844.0  alta_igualdad_alta_brecha       True
✓ OK

Guardado: ../data/processed/v4_scatter_nordica.csv


## 7. Resumen

In [10]:
print('=' * 60)
print('RESUMEN V4 — SCATTER IGUALDAD FORMAL')
print('=' * 60)
print()
print(f'Países en el scatter: {len(v4_out)}')
print(f'Correlación EIGE ↔ BRECHA_GNDR: {corr:.3f}')
print(f'Medianas — EIGE: {mediana_eige:.1f} | BRECHA: {mediana_brecha:.3f}')
print()
print('Distribución por cuadrante:')
for q, cnt in v4_out['cuadrante'].value_counts().items():
    paises = sorted(v4_out[v4_out['cuadrante']==q]['cntry'].tolist())
    print(f'  {q}: {cnt} países — {paises}')
print()
es = v4_out[v4_out['cntry']=='ES'].iloc[0]
print(f'España: EIGE={es.eige_index} | BRECHA={es.BRECHA_GNDR} | cuadrante={es.cuadrante}')
print()
print('Configuración Flourish:')
print('  Tipo: Scatter')
print('  Eje X: eige_index (igualdad formal 0-100)')
print('  Eje Y: BRECHA_GNDR (brecha de bienestar)')
print('  Etiqueta: country_name')
print('  Color: cuadrante (4 categorías)')
print('  Tamaño: n_total (o fijo)')
print('  España: es_espana=True → destacada')

RESUMEN V4 — SCATTER IGUALDAD FORMAL

Países en el scatter: 22
Correlación EIGE ↔ BRECHA_GNDR: -0.134
Medianas — EIGE: 61.0 | BRECHA: 0.105

Distribución por cuadrante:
  alta_igualdad_alta_brecha: 6 países — ['BE', 'ES', 'FR', 'IT', 'NL', 'PT']
  baja_igualdad_baja_brecha: 6 países — ['BG', 'EE', 'HU', 'LT', 'LV', 'PL']
  alta_igualdad_baja_brecha: 5 países — ['AT', 'DE', 'FI', 'IE', 'SE']
  baja_igualdad_alta_brecha: 5 países — ['CY', 'GR', 'HR', 'SI', 'SK']

España: EIGE=70.9 | BRECHA=0.205 | cuadrante=alta_igualdad_alta_brecha

Configuración Flourish:
  Tipo: Scatter
  Eje X: eige_index (igualdad formal 0-100)
  Eje Y: BRECHA_GNDR (brecha de bienestar)
  Etiqueta: country_name
  Color: cuadrante (4 categorías)
  Tamaño: n_total (o fijo)
  España: es_espana=True → destacada
